Import necessary libraries

In [ ]:
import polars as pl
import plotly.express as px

import plotly.io as pio

pio.renderers.default = "notebook"

## 1. Data Cleaning and Preprocessing

To simplify data preprocessing process, we will choose the `Spotify_1Million_Tracks` dataset as our base, then merge other datasets with it.

In [ ]:
df = pl.scan_csv("../datasets/spotify-1million-tracks/spotify_data.csv").drop("")
df_30000_songs = pl.scan_csv("../datasets/30000-spotify-songs/spotify_songs.csv")
df_2023 = pl.scan_csv("../datasets/spotify-dataset-2023/spotify_data_12_20_2023.csv")
df_tracks_genre = pl.scan_csv(
    "../datasets/spotify-tracks-genre-dataset/train.csv"
).drop("Unnamed: 0")

Let's check the columns in all datasets to ensure consistency. We will only keep columns that are common across all datasets but with bias toward `Spotify_1Million_Tracks` since it has the most comprehensive data.

In [ ]:
print(df.collect_schema().names())

In [ ]:
print(df_30000_songs.collect_schema().names())

In [ ]:
print(df_2023.collect_schema().names())

In [ ]:
print(df_tracks_genre.collect_schema().names())

Based on this, we have some initial comments:
1. All datasets contain the same fundamental data (audio features, identifiers, popularity) but use different column names (e.g., `artist_name` vs. `track_artist` vs. `artist_0`). We cannot simply merge or concatenate them without renaming columns to a single "master" schema first, in our case, the schema of `Spotify_1Million_Tracks`.
2. `df_30000_songs` uses `playlist_genre`, `df_tracks_genre` uses `track_genre`, and `df_2023` splits genres into `genre_{0, 1, 2, 3, 4}`. We will have to consolidate these into a single `genre` column.
3. Since we handle multiple datasets, there will be duplicate tracks (same `track_id`) across datasets. We decide to keep only the one with the highest popularity score.

In [ ]:
target_columns = [
    "track_id",
    "track_name",
    "artist_name",
    "popularity",
    "year",
    "genre",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
    "time_signature",
]  # This is basically all columns from `df`

Standardize column names across datasets and merge them into a single DataFrame `combined_df`.

In [ ]:
def standardize(df):
    return df.with_columns(
        [
            pl.col("track_id").cast(pl.String),
            pl.col("popularity").cast(pl.Float64, strict=False),
        ]
    )

In [ ]:
df_30000_songs = df_30000_songs.with_columns(
    year=pl.col("track_album_release_date").str.to_date(strict=False).dt.year(),
    genre=pl.col("playlist_genre").fill_null(pl.col("playlist_subgenre")),
).rename(
    {
        "track_artist": "artist_name",
        "track_popularity": "popularity",
    }
)
df_30000_songs = standardize(df_30000_songs).select(
    [col for col in target_columns if col in df_30000_songs.collect_schema().names()]
)

In [ ]:
# Define logic to extract first genre from string "['pop', 'rock']"
# Regex: matches text inside the first set of single or double quotes
parsed_artist_genre = pl.col("artist_genres").str.extract(r"['\"]([^'\"]+)['\"]", 1)

# Apply transformations
df_2023 = df_2023.with_columns(
    # Coalesce tries genre_0..4, then falls back to the parsed artist_genre
    genre=pl.coalesce([pl.col(f"genre_{i}") for i in range(5)] + [parsed_artist_genre]),
    year=pl.col("release_year").cast(pl.Int32, strict=True),
).rename(
    {
        "artist_0": "artist_name",
        "track_popularity": "popularity",
    }
)
df_2023 = standardize(df_2023).select(
    [col for col in target_columns if col in df_2023.collect_schema().names()]
)

In [ ]:
df_tracks_genre = df_tracks_genre.rename(
    {
        "artists": "artist_name",
        "track_genre": "genre",
    }
)
df_tracks_genre = standardize(df_tracks_genre).select(
    [col for col in target_columns if col in df_tracks_genre.collect_schema().names()]
)

In [ ]:
df = standardize(df.with_columns(pl.col("year").cast(pl.Int32, strict=True)))
df = df.select([col for col in target_columns if col in df.collect_schema().names()])

In [ ]:
combined_df = pl.concat([df, df_30000_songs, df_2023, df_tracks_genre], how="diagonal")

In [ ]:
# Number of rows in each dataset before merging
print(f"Rows in df: {df.select(pl.len()).collect().item()}, Rows in df_30000_songs: {df_30000_songs.select(pl.len()).collect().item()}, Rows in df_2023: {df_2023.select(pl.len()).collect().item()}, Rows in df_tracks_genre: {df_tracks_genre.select(pl.len()).collect().item()}")
print(f"Total rows before removing duplicates: {combined_df.select(pl.len()).collect().item()}")

Let's first check dupilcate based on `track_id`.

In [ ]:
duplicated = (
    combined_df.select(pl.len()).collect().item()
    - combined_df.select("track_id").collect().n_unique()
)
print(f"Number of duplicated track_id entries: {duplicated}")

In [ ]:
combined_df.null_count().collect()

We will try to handle both missing data and duplicates in the next step.

In [ ]:
# We first sort by `track_id` and `popularity` (descending), then drop duplicates keeping the first occurrence (highest popularity)
combined_df = combined_df.sort(
    ["track_id", "popularity"], descending=[False, True]
).unique(subset=["track_id"], keep="first")

In [ ]:
duplicate_count = combined_df.select(
    pl.len() - pl.col("track_id").n_unique()
).collect().item()

print(f"Number of duplicates after processing: {duplicate_count}")

In [ ]:
audio_cols = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
]

combined_df = (
    combined_df.with_columns(
        # Fill them with -1 and convert to integer type
        pl.col("year").fill_null(-1).cast(pl.Int32),
        pl.col("genre").fill_null("Unknown"),
        pl.col("artist_name").fill_null("Unknown"),
        pl.col("track_name").fill_null("Unknown"),
        # Fill missing time_signature with 4.0 (most common time signature is 4/4)
        pl.col("time_signature").fill_null(4.0),
    )
    # Drops row if ANY of these columns are null (matches your pandas code behavior)
    .drop_nulls(subset=audio_cols + ["popularity"])
)

In [ ]:
combined_df.null_count().collect()

In [ ]:
print(f"Total rows before removing duplicates: {df.select(pl.len()).collect().item() + df_30000_songs.select(pl.len()).collect().item() + df_2023.select(pl.len()).collect().item() + df_tracks_genre.select(pl.len()).collect().item()}")
print(f"Total rows after removing duplicates and missing data: {combined_df.select(pl.len()).collect().item()}")

For later analysis purpose, we also mapping genres to broader categories. We will define a mapping dictionary for this purpose.

In [ ]:
import json

# 1. load the genre direct mapping dictionary
mapping = json.load(open("./genre_mapping.json"))

# 2. regex patterns for the fallback logic
rock_pat = r"metal|rock|punk|emo|grunge|hardcore|indie|goth"
elec_pat = r"house|techno|trance|dubstep|edm|electro|dance|synth|rave|bass"
hiphop_pat = r"rap|hip hop|trap|drill|r&b|soul|funk|urban"
latin_pat = r"latin|reggaeton|salsa|bachata|mariachi|samba|tropical|cumbia"
reggae_pat = r"reggae|dub|dancehall"
folk_pat = r"country|folk|acoustic|bluegrass"
jazz_pat = r"jazz|blues"
classic_pat = r"classical|piano|orchestra|film|movie|score|soundtrack|violin"
ambient_pat = r"ambient|chill|sleep|new age|nature"
relig_pat = r"gospel|christian|worship"

# 3. create polars Expression
# create the cleaned column to avoid repetition
clean_g = pl.col("genre").str.strip_chars().str.to_lowercase()

combined_df = combined_df.with_columns(
    super_genre=(
        pl.when(clean_g.is_in(mapping.keys()))
        .then(clean_g.replace(mapping))  # priority 1: Exact Dict Match
        .when(clean_g.str.contains(rock_pat))
        .then(pl.lit("Rock & Metal"))  # priority 2: Rock keywords
        .when(clean_g.str.contains(elec_pat))
        .then(pl.lit("Electronic"))  # priority 3: Electronic keywords
        .when(clean_g.str.contains(hiphop_pat))
        .then(pl.lit("Hip-Hop & R&B"))
        .when(clean_g.str.contains(r"pop|idol"))
        .then(pl.lit("Pop"))
        .when(clean_g.str.contains(latin_pat))
        .then(pl.lit("Latin"))
        .when(clean_g.str.contains(reggae_pat))
        .then(pl.lit("Reggae & Dub"))
        .when(clean_g.str.contains(folk_pat))
        .then(pl.lit("Folk & Country"))
        .when(clean_g.str.contains(jazz_pat))
        .then(pl.lit("Jazz & Blues"))
        .when(clean_g.str.contains(classic_pat))
        .then(pl.lit("Classical & Soundtrack"))
        .when(clean_g.str.contains(ambient_pat))
        .then(pl.lit("Ambient & Chill"))
        .when(clean_g.str.contains(relig_pat))
        .then(pl.lit("Religious"))
        .otherwise(pl.lit("Other/Unknown"))  # default case
    )
)

In [ ]:
print("Sample rows:")
combined_df.head(5).collect()

In [ ]:
# Save our final preprocessed dataset
combined_df.sink_csv("../datasets/spotify_tracks_preprocessed.csv")

Summary of columns in our combined dataset (based on the **official Spotify Web API** documentation):

| Column | Data Type | Description | Range / Values |
| :--- | :--- | :--- | :--- |
| **track_id** | String | The unique ID for the track generated by Spotify. This is the primary key for merging datasets. | Alphanumeric (e.g., `11dFghVXANMlKmJXsNCbN5`) |
| **artist_name** | String | The name of the primary artist(s) who performed the track. | Text |
| **track_name** | String | The title of the track. | Text |
| **popularity** | Integer | A value between 0 and 100, with 100 being the most popular. It is calculated by an algorithm based on the total number of plays and how recent those plays are. | `0` to `100` |
| **year** | Integer | The year the track was released. | e.g., `2023`, `1998` |
| **genre** | String | The genre associated with the track (often derived from the playlist or artist). | Text (e.g., `Pop`, `Rock`) |
| **danceability** | Float | Describes how suitable a track is for dancing based on tempo, rhythm stability, beat strength, and overall regularity. | `0.0` (Least danceable) to `1.0` (Most danceable) |
| **energy** | Float | A perceptual measure of intensity and activity. Energetic tracks feel fast, loud, and noisy (e.g., Death Metal), while low energy tracks feel calm (e.g., Bach Prelude). | `0.0` to `1.0` |
| **key** | Integer | The key the track is in. Integers map to pitches using standard [Pitch Class notation](https://en.wikipedia.org/wiki/Pitch_class). <br> `0` = C, `1` = C♯/D♭, `2` = D, etc. | `0` to `11` ( `-1` if no key detected) |
| **loudness** | Float | The overall loudness of a track in decibels (dB). Loudness values are averaged across the entire track. | Typically `-60.0` dB to `0.0` dB |
| **mode** | Integer | Indicates the modality (major or minor) of a track, the type of scale from which its melodic content is derived. | `1` = Major, `0` = Minor |
| **speechiness** | Float | Detects the presence of spoken words. <br>• `> 0.66`: Entirely spoken (podcast, poetry) <br>• `0.33 - 0.66`: Mix of music & speech (Rap) <br>• `< 0.33`: Music | `0.0` to `1.0` |
| **acousticness** | Float | A confidence measure of whether the track is acoustic. | `0.0` (Not acoustic) to `1.0` (High confidence acoustic) |
| **instrumentalness**| Float | Predicts whether a track contains no vocals. "Ooh" and "aah" sounds are treated as instrumental. Rap or spoken word tracks are "vocal". | `> 0.5` is intended to be instrumental. Closer to `1.0` is higher confidence. |
| **liveness** | Float | Detects the presence of an audience in the recording. Higher values represent an increased probability that the track was performed live. | `> 0.8` is strongly likely live. |
| **valence** | Float | A measure of the musical "positiveness" conveyed by a track. High valence sounds happy/cheerful; low valence sounds sad/depressed/angry. | `0.0` (Negative) to `1.0` (Positive) |
| **tempo** | Float | The overall estimated tempo of a track in beats per minute (BPM). | e.g., `120.0`, `94.5` |
| **duration_ms** | Integer | The duration of the track in milliseconds. | e.g., `234000` (approx 3m 54s) |
| **time_signature** | Integer | An estimated time signature. The time signature (meter) specifies how many beats are in each bar (or measure). | `3` to `7` (The most common is `4`) |

## 2. EDA

### 2.1 Overview

#### 2.1.1 Popularity Distribution

In [ ]:
fig = px.histogram(
    combined_df.collect(),
    x="popularity",
    nbins=100,
    title="Distribution of Track Popularity (0-100)",
    labels={"popularity": "Popularity Score"},
    color_discrete_sequence=["#1DB954"],  # Spotify Green
)

fig.update_layout(bargap=0.1, xaxis_title="Popularity Score", yaxis_title="#Tracks")

p80 = combined_df.select("popularity").quantile(0.8).collect().row(0)[0]
fig.add_vline(
    x=p80,
    line_width=2,
    line_dash="dot",
    line_color="red",
    annotation_text=f"P80: {p80:.1f}",
    annotation_position="top right"
)
p90 = combined_df.select("popularity").quantile(0.9).collect().row(0)[0]
fig.add_vline(
    x=p90,
    line_width=2,
    line_dash="dot",
    line_color="red",
    annotation_text=f"P80: {p90:.1f}",
    annotation_position="top right"
)

fig.show()

This chart reveals a classic "Long Tail" distribution (or Pareto distribution).

Key Observations:
- The massive column at 0 popularity (nearly 400k tracks, or ~30-40% of data): This represents the "Long Tail" of Spotify. These are likely tracks that have very few recent streams, are very old, or are from obscure artists. In the Spotify algorithm, popularity is calculated based on recent play counts. If a song hasn't been played lately, its score drops to 0.
- Sharp decline immediately after 0: Even getting a score of 10-20 puts a track ahead of a huge portion of the library.
- The bars above a score of 80 are barely visible: This shows that "viral" status is statistically very difficult to achieve. A song with a popularity of 50 is actually doing quite well relative to the global catalog.

#### 2.1.2 Genre Distribution

In [ ]:
genre_counts = (
    combined_df.filter(pl.col("super_genre") != "Other/Unknown")
    .select(
        [
            pl.col("super_genre"),
            pl.len().alias("count"),
        ]
    )
    .collect()
    .group_by("super_genre")
    .agg(pl.col("count").count())
    .sort("count", descending=True)
)

# 2. Plot
fig = px.bar(
    genre_counts,
    x="count",
    y="super_genre",
    orientation="h",
    title="Distribution of Super Genres",
    text="count",
    color="count",
    color_continuous_scale="Greens",
)

fig.update_layout(
    yaxis=dict(autorange="reversed"),  # Top genre at the top
    xaxis_title="Number of Tracks",
    yaxis_title="Genre",
)

fig.show()

Key observations:

-   The dominance of **Rock & Metal** (292k) and **Electronic** (224k) reflects the massive volume of historical production and sub-genres in these categories, rather than current streaming popularity.
-   The strong presence of **Latin** (118k) and **Pop** (179k) confirms that merging the `df_2023` dataset effectively injected global trends into the catalog-heavy base.

#### 2.1.3 Song Duration Distribution

In [ ]:
# filter out extreme outliers (> 15 mins / 900,000ms)
df_dur = combined_df.filter(pl.col("duration_ms") < 900000).with_columns(
    duration_min=pl.col("duration_ms") / 60000
).collect()

fig = px.histogram(
    df_dur,
    x="duration_min",
    nbins=100,
    title="Distribution of Song Duration (Minutes)",
    labels={"duration_min": "Duration (Minutes)"},
    color_discrete_sequence=["#191414"],  # Spotify Black
)

mean_val = df_dur["duration_min"].mean()
fig.add_vline(
    x=mean_val,
    line_dash="dash",
    line_color="#1DB954",
    annotation_text=f"Mean: {mean_val:.2f} min",
)

fig.update_layout(
    xaxis_title="Duration (Minutes)", yaxis_title="Count of Tracks", bargap=0.1
)

fig.show()

Key observations:
- The majority of songs cluster between 2 to 5 minutes, with a peak around 3 to 4 minutes, which aligns with traditional pop song lengths.
- There is a noticeable drop-off in frequency for songs longer than 5 minutes, indicating that longer tracks are less common in the dataset.
- A small number of outliers exist with durations exceeding 10 minutes, which may represent live recordings, extended mixes, or classical pieces.

#### 2.1.4 Global Distribution of Audio Features

In [ ]:
features_to_plot = ["danceability", "energy", "valence", "acousticness"]

# unpivot() automatically drops columns not specified in 'index' or 'on'
df_melt = combined_df.unpivot(
    on=features_to_plot, variable_name="Feature", value_name="Value"
).collect()

fig = px.histogram(
    df_melt,
    x="Value",
    color="Feature",
    barmode="overlay",
    title="Global Distribution of Audio Features",
    opacity=0.6,
    nbins=100,
    color_discrete_sequence=px.colors.qualitative.Bold,
    labels={"Value": "Score (0 to 1)", "count": "Count"},
)

fig.update_layout(
    xaxis_title="Score (0.0 to 1.0)", yaxis_title="Count of Tracks", bargap=0.05
)

fig.show()

Key observations:

- Massive Acousticness at 0.0: ~400,000+ tracks have essentially zero acousticness. This is the definitive proof that dataset is dominated by electric instruments (Rock/Metal guitars) and synthesized sounds (Electronic/Pop). Acoustic music (Classical, Folk) is present (the small bump at the far right), but it is a minority.
- Energy skewed very high: The energy distribution is heavily right-skewed, peaking near 0.9. This indicates that most tracks are loud, fast, and intense, typical of Rock, Metal, and Electronic genres.
- Danceability is normally distributed: Unlike other features, danceability forms a nice bell curve centered around 0.55. This suggests a good mix of rhythmic styles, from slow ballads to upbeat dance tracks.
- Valence is relatively flat: The valence distribution is surprisingly even, with a slight lean towards lower values. This indicates a balance between happy and sad tracks, but with a tendency towards more serious or aggressive tones, likely due to the Rock/Metal influence.

### 2.2 Correlations

#### 2.2.1 Correlation Matrix

In [ ]:
# select the numeric columns of interest
corr_cols = [
    "popularity",
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

corr_matrix = combined_df.select(corr_cols).collect().corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Matrix of Audio Features",
    color_continuous_scale="RdBu_r",  # Red=Positive, Blue=Negative
    zmin=-1,
    zmax=1,
)

fig.update_layout(xaxis_title="Features", yaxis_title="Features")

fig.show()

Key observations:

* Loudness & Energy (0.77): A very strong positive correlation, as expected. Louder tracks tend to be more energetic.
* Acousticness & Energy (-0.73): A strong negative correlation. Acoustic tracks are generally less energetic.
* Danceability & Valence (0.51): Moderately positive correlation. More danceable tracks tend to be happier.
* On Popularity correlations:
    * The "strongest" predictor is Loudness (0.16), suggesting louder songs have a slight advantage.
    * Instrumentalness (-0.16) is negative, confirming that vocal tracks are generally more popular than instrumental ones.
    * We might not predict a hit song using audio features alone.

#### 2.2.2 Valence vs. Energy

In [ ]:
fig = px.density_heatmap(
    combined_df.collect(),
    x="valence",
    y="energy",
    nbinsx=50,
    nbinsy=50,
    title="The Mood Map: Valence vs. Energy",
    color_continuous_scale="Magma",  # Dark = Low Density, Bright = High Density
)

# 2. Add Annotations for the Quadrants
fig.add_annotation(
    x=0.1,
    y=0.9,
    text="TURBULENT/ANGRY<br>(Metal/Hardcore)",
    showarrow=False,
    font=dict(color="white"),
)
fig.add_annotation(
    x=0.9,
    y=0.9,
    text="HAPPY/PARTY<br>(Pop/Dance)",
    showarrow=False,
    font=dict(color="white"),
)
fig.add_annotation(
    x=0.1,
    y=0.1,
    text="SAD/DEPRESSING<br>(Ballads)",
    showarrow=False,
    font=dict(color="white"),
)
fig.add_annotation(
    x=0.9,
    y=0.1,
    text="PEACEFUL/CALM<br>(Acoustic/Jazz)",
    showarrow=False,
    font=dict(color="white"),
)

fig.update_layout(
    xaxis_title="Valence (0 = Negative, 1 = Positive)",
    yaxis_title="Energy (0 = Calm, 1 = Intense)",
)

fig.show()

Key observations:

- Bottom-Left: The brightest spot on the entire map is in the extreme bottom-left corner (Low Energy, Low Valence). There is a massive concentration of dark, slow, and depressing music. This is likely the "Ambient", "Classical", and "Sleep" genres we saw earlier. It suggests a huge portion of the library is "functional" music (for studying/sleeping) rather than "emotional" music.
- Top Band: The distinct horizontal streak across the top (Energy > 0.8). It stretches across the entire valence spectrum. This is Rock, Metal, and Electronic core. This band is thicker on the Left (Angry/Turbulent) than on the Right (Happy). Our dataset has more "Aggressive High Energy" tracks than "Happy High Energy" tracks.
- Bottom-Right: The "Peaceful/Calm" area (Low Energy, High Valence) is almost empty (black). It is very rare to find a song that is very slow but also very happy. "Chill" music tends to be melancholic or neutral, not joyful.

### 2.3 Temporal Trends

#### 2.3.1 Loudness

In [ ]:
loudness_trend = (
    combined_df.filter((pl.col("year") >= 1950) & (pl.col("year") <= 2023))
    .collect()
    .group_by("year")
    .agg(pl.col("loudness").mean())
    .sort("year")
)

fig = px.line(
    loudness_trend,
    x="year",
    y="loudness",
    title="The Loudness War: Average Loudness (dB) by Year",
    markers=True,
    color_discrete_sequence=["#FF4B4B"],  # Red for "Warning/Loud"
)

fig.add_hline(y=0, line_dash="dot", annotation_text="0 dB (Max Limit)")

fig.update_layout(xaxis_title="Year", yaxis_title="Average Loudness (dB)")

fig.show()

Key observations: The data shows two distinct behaviors before and after roughly 1995.

- Before 1995: The line is volatile and jagged, fluctuating randomly between -13 dB and -17 dB. There is no consistent directional trend, suggesting that "loudness" was not a primary mastering goal during this era.
- After 1995: The volatility disappears and is replaced by a smooth, linear upward trend. This indicates a systematic, industry-wide shift in how music was processed.

1. Significant Amplitude Increase (6 dB Gain)
Between 1990 and 2010, the average loudness increased from approximately -15 dB to -9 dB.
In audio physics, an increase of 6 dB is roughly equivalent to doubling the signal amplitude. This means the average track in 2010 was physically twice as "loud" (in terms of signal pressure) as a track from 1990.
1. The "Saturation Point" and Decline
The trend hits a hard ceiling around 2010, hovering flat at -9 dB for nearly a decade. The data suggests the industry hit a physical limit where music could not get louder without degrading quality.
Current Trend: The tail end of the chart (2020–2023) shows a confirmed downward trend (dropping to -11 dB), marking the first consistent decrease in loudness in the entire 70-year dataset.

#### 2.3.2 Long Song

In [ ]:
duration_trend = (
    combined_df.filter((pl.col("year") >= 1950) & (pl.col("year") <= 2023))
    .collect()
    .group_by("year")
    .agg(pl.col("duration_ms").mean())
    .sort("year")
)
duration_trend = duration_trend.with_columns(duration_min=pl.col("duration_ms") / 60000)

fig = px.line(
    duration_trend,
    x="year",
    y="duration_min",
    title="Average Song Duration by Year (1950-2023)",
    markers=True,
    color_discrete_sequence=["#1DB954"],  # Spotify Green
)

fig.add_hline(
    y=duration_trend.select("duration_min").max().row(0)[0],
    line_dash="dot",
    annotation_text="Peak average song duration",
)

fig.update_layout(xaxis_title="Year", yaxis_title="Average Duration (Minutes)")

fig.show()

Key observations: The most dominant feature is the dramatic, linear decline in song length starting exactly around 2010. The average drops from a peak of ~4.4 minutes (2010) to ~3.4 minutes (2023). This is a nearly 1-minute loss (25%) in just over a decade, perfectly correlating with the rise of Spotify and streaming economics (where artists are paid per play, incentivizing shorter tracks) and the "TikTok effect" (shorter attention spans).

### 2.4 Popularity Analysis

#### 2.4.1 Popularity by Super Genre

In [ ]:
pop_by_genre = (
    combined_df.collect()
    .group_by("super_genre")
    .agg(pl.col("popularity").median())
    .sort("popularity")
)

fig = px.bar(
    pop_by_genre,
    x="popularity",
    y="super_genre",
    orientation="h",
    title="Median Popularity by Super Genre",
    text="popularity",
    color="popularity",
    color_continuous_scale="Greens",
)

fig.update_layout(xaxis_title="Median Popularity Score", yaxis_title="Genre")

fig.show()

Key observations:

Hip-Hop & R&B secures the highest median popularity, confirming its status as the dominant engine of the 2023 streaming economy where even average tracks enjoy high baseline consumption. Jazz & Blues surprisingly ranks second, driven by the massive rise of "functional" listening (e.g., study or coffee shop playlists) that keeps older catalog tracks consistently active. In contrast, high-volume genres like Latin and Pop show much lower medians, revealing a "winner-takes-all" market where a few viral hits dominate while the vast "long tail" of tracks remains largely unstreamed.

People might actively engage with Hip-Hop (High Median), they passively consume Jazz (High Median), and they selectively binge the top 1% of Pop/Latin while ignoring the other 99% (Low Median).

#### 2.4.2 Danceability Factor

In [ ]:
dance_trend = (
    combined_df.with_columns(dance_bin=pl.col("danceability").round(1))
    .collect()
    .group_by("dance_bin")
    .agg(pl.col("popularity").median())
    .sort("dance_bin")
)

fig = px.area(
    dance_trend,
    x="dance_bin",
    y="popularity",
    title="Danceability vs. Median Popularity (2023 Snapshot)",
    markers=True,
    color_discrete_sequence=["#9400D3"],  # Purple
)

fig.update_layout(
    xaxis_title="Danceability Score (0.0 = Static, 1.0 = Highly Danceable)",
    yaxis_title="Median Popularity Score",
)

fig.show()

Key observations:
- The median popularity for non-danceable tracks is near zero.
- The highest sustained popularity (Median ~14) occurs in the mid-range, not the high-end.
- After a slight dip, popularity spikes again at 0.9.

#### 2.4.3 Artist vs Track Popularity

In [ ]:
hits_df = (
    combined_df.filter(
        (pl.col("popularity") > 70) & (pl.col("artist_name") != "Unknown")
    )
    .collect()
    .group_by("artist_name")
    .agg(
        pl.col("popularity").count().alias("num_hits"),
        pl.col("popularity").mean().alias("popularity"),
    )
    .with_columns(
        artist_category=pl.when(pl.col("num_hits") == 1)
        .then(pl.lit("One-Hit Wonder (1 Hit)"))
        .otherwise(pl.lit("Superstar (Multiple Hits)"))
    )
)

fig = px.box(
    hits_df,
    x="artist_category",
    y="popularity",
    title="Quality vs. Quantity: Do One-Hit Wonders have bigger hits?",
    color="artist_category",
    color_discrete_map={
        "One-Hit Wonder (1 Hit)": "#FF4B4B",  # Red
        "Superstar (Multiple Hits)": "#1DB954",  # Spotify Green
    },
    points="outliers",
)

fig.update_layout(
    xaxis_title="Artist Category",
    yaxis_title="Popularity of the Hit Song",
    showlegend=False,
)

fig.show()

print(
    "Number of One-Hit Wonders identified:",
    hits_df.filter(pl.col("artist_category") == "One-Hit Wonder (1 Hit)").height,
)
print(
    "Number of Superstars identified:",
    hits_df.filter(pl.col("artist_category") == "Superstar (Multiple Hits)").height,
)

Key observations: This chart shows that Superstars (Green) consistently outperform One-Hit Wonders (Red).

- The median popularity for Superstars (~76) is visibly higher than for One-Hit Wonders (~74).
- While the average One-Hit Wonder scores lower, the Red Outliers reaching 100 popularity confirm that a viral sensation (likely a TikTok trend) can rival the biggest stars in the world for a brief moment, but these are statistically rare exceptions.
- The data identifies 1,641 One-Hit Wonders but only 947 Superstars. This 1.75:1 ratio highlights it is almost twice as hard to sustain a career at the top as it is to break into it once.

## 3. Hypothesis Formation

1.   **Hypothesis:** Songs released in the "Streaming Era" (2015–2023) are significantly shorter than songs released in the "Digital Download Era" (2000–2010).

     *   **Statistical Test:** **Two-Sample t-test** (or Mann-Whitney U Test).
     *   **Null Hypothesis ($H_0$):** There is no difference in the mean duration of songs between the 2000–2010 period and the 2015–2023 period.
     *   **Observation:** Chart _2.3.2 Long Song_ showed a steep linear decline in average song duration starting around 2010, dropping from ~4.3 minutes to ~3.4 minutes.

2.   **Hypothesis:** The median popularity of "Hip-Hop & R&B" tracks is statistically higher than the median popularity of "Rock & Metal" tracks in the 2023 ecosystem.

     *   **Statistical Test:** **Mann-Whitney U Test** (Comparing two independent groups with non-normal distributions).
     *   **Null Hypothesis ($H_0$):** The median popularity distribution of Hip-Hop & R&B is equal to that of Rock & Metal.
     *   **Observation:** Chart _2.4.1 Popularity by Super Genre_ revealed that Hip-Hop & R&B had the highest median popularity (25), while Rock & Metal was significantly lower (16).

3.   **Hypothesis:** Tracks classified as **"Vocal-Centric"** (`instrumentalness` < 0.1) have a statistically higher median popularity than **"Instrumental"** tracks (`instrumentalness` > 0.5).

     *   **Statistical Test:** **Mann-Whitney U Test** (Non-parametric test for two independent groups).
     *   **Null Hypothesis ($H_0$):** There is no difference in the median popularity between vocal-centric tracks and instrumental tracks.
     *   **Observation:** This validates the "Personality" aspect of the music business. If rejected, it proves that "Background Music" (utility listening) has become just as commercially viable in the streaming era as traditional "Foreground Music" (active listening).

4.   **Hypothesis:** Tracks falling into the "Rhythmic/Rap" speechiness range ($0.33 - 0.66$) have a higher mean popularity than tracks in the "Melodic/Music" range ($< 0.33$).

     *   **Statistical Test:** **Welch’s t-test** (assuming unequal variances/sample sizes between the massive "Music" group and the "Rap" group).
     *   **Null Hypothesis ($H_0$):** The mean popularity of "Rhythmic/Rap" tracks is less than or equal to that of "Melodic/Music" tracks.
     *   **Observation:** This quantifies the dominance of Hip-Hop culture. If true, it suggests that adding rhythmic speech elements to a track is currently the most effective way to increase its commercial viability, more so than traditional singing.

5.   **Hypothesis:**  There is a non-linear (quadratic) relationship between Duration and Popularity, where popularity peaks for songs between **2:30 (150,000ms)** and **3:30 (210,000ms)** and significantly decreases for songs shorter than 2:00 or longer than 4:00.

     *   **Statistical Test:** **Polynomial Regression** (Degree 2) or **ANOVA** (by binning duration into Short/Optimal/Long).
     *   **Null Hypothesis ($H_0$):** Popularity is distributed evenly across all duration bins, or the relationship is strictly linear.
     *   **Observation:** This detects "Optimization" in the streaming economy. If the peak is strictly between 2:30–3:30, it confirms that artists are engineering song lengths to maximize stream counts without making the song so short it feels incomplete to the listener.